# 10 · Gradient-based geometry search

Chapter 09 searched for fault geometry by trying many values and picking the
best. That works, but it is wasteful: a grid spends most of its effort
evaluating models far from the answer, and its cost explodes as the number of
unknown geometry parameters grows. This chapter uses **automatic
differentiation** to compute the exact slope of the misfit surface, so an
optimizer can walk straight downhill toward the minimum. It is the same
variable-projection idea as Chapter 09, made faster.

**Learning objectives**

By the end of the chapter, you will be able to:

- explain, at a practical level, what automatic differentiation does
- run a differentiable geometry search with `geodef.invert.geometry_search`
- recover several geometry parameters at once from a poor starting guess
- read the first-order (Gauss-Newton) error bars it returns, and know when to
  trust them

**Prerequisites:** Chapter 09; the optional `geodef[jax]` dependency
**Estimated time:** about 60 minutes

> **Optional dependency.** This chapter needs JAX, installed with
> `pip install "geodef[jax]"`. If JAX is absent the chapter is skipped.

GeoDef's axes, signs, and units are summarized in
[the conventions guide](../docs/conventions.md). New terms are introduced in
words here and collected in [the glossary](../docs/glossary.md).

## 1. What automatic differentiation buys us

To walk downhill, an optimizer needs the **gradient**: how much the misfit
changes for a small change in each geometry parameter. Chapter 09's grid search
never computed a gradient; it just sampled. A local optimizer *can* estimate
one by finite differences, nudging each parameter a little and measuring the
response, but that costs an extra evaluation per parameter and introduces
rounding error.

**Automatic differentiation** avoids both problems. It applies the chain rule
to the actual sequence of operations the forward model performs, from the Okada
kernel through the linear slip solve, and returns the *exact* derivative for
essentially the cost of one forward evaluation. JAX, the library GeoDef uses
for this, also compiles the whole calculation to fast machine code the first
time it runs.

The pieces fit together like this: variable projection (Chapter 09) reduces the
problem to a misfit that depends only on geometry, JAX differentiates that
misfit exactly, and the optimizer L-BFGS-B follows the gradient to the minimum.
`geodef.invert.geometry_search` wires all three together.

## 2. Switch on the JAX backend and rebuild the scenario

Automatic differentiation requires the JAX backend, so we select it first.
Everything else is the shared megathrust scenario from Chapter 03. We keep
float64 precision (JAX's default here), which matters because geodetic
coordinates involve delicate cancellation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

geodef.backend.set_backend("jax")   # exact gradients need the JAX backend

%matplotlib inline

rng = np.random.default_rng(0)

In [ ]:
# The shared megathrust, true slip, GNSS network, and 1 cm noise.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,
    strike=315.0, dip=15.0,
    length=180e3, width=90e3,
    n_length=12, n_width=6,
)
N = fault.n_patches
n_length, n_width = fault.grid_shape
along = np.arange(N) % n_length
down = np.arange(N) // n_length
bump = np.exp(-(((along - (n_length - 1) / 2) / 3.0) ** 2
                + ((down - n_width / 2) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

grid_lon, grid_lat = np.meshgrid(np.linspace(98.5, 101.5, 8),
                                 np.linspace(-3.6, -0.4, 8))
station_lon, station_lat = grid_lon.ravel(), grid_lat.ravel()
n_stations = station_lon.size

G_full = fault.greens_matrix(station_lat, station_lon)
sigma = 0.01
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, 3 * n_stations)
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=d_obs[0::3], north=d_obs[1::3], up=d_obs[2::3],
    sigma_east=np.full(n_stations, sigma),
    sigma_north=np.full(n_stations, sigma),
    sigma_up=np.full(n_stations, sigma),
)
print(f"{N} patches, {n_stations} stations, backend = {geodef.backend.get_backend()}")

## 3. Recover one parameter from a bad start

We repeat Chapter 09's task, recovering dip, but now with gradients. The search
takes a starting geometry vector and a list of which entries to optimize. The
geometry vector is

$$ \boldsymbol{\theta} = [\,e_0,\ n_0,\ \text{depth},\ \text{strike},\
   \text{dip},\ \text{length},\ \text{width}\,], $$

with position measured in a local frame anchored at `(ref_lat, ref_lon)`. Here
`free=["dip"]` optimizes only the dip, starting from a deliberately wrong 30
degrees.

The first call compiles the differentiable model, which takes some seconds;
later calls with the same problem shape reuse the compiled code and run in
milliseconds.

In [ ]:
theta0 = np.array([0.0, 0.0, 25e3, 315.0, 30.0, 180e3, 90e3])   # start dip 30

result_dip = geodef.invert.geometry_search(
    theta0, gnss,
    ref_lat=-2.0, ref_lon=100.0,
    free=["dip"],
    bounds={"dip": (5.0, 45.0)},
    n_length=n_length, n_width=n_width,
    components="dip",
    regularization="laplacian", regularization_strength=1.0,
)
dip_sigma = np.sqrt(result_dip.theta_cov[0, 0])
print(f"recovered dip: {result_dip.theta[4]:.2f} +/- {dip_sigma:.2f} degrees  (true 15.0)")
print(f"reduced chi-squared: {result_dip.reduced_chi2:.3f}, "
      f"iterations: {result_dip.n_iterations}")

From a start 15 degrees away, the search converges in a handful of iterations
and returns an error bar alongside the estimate. That error bar deserves a
careful look, which we come to in Section 5.

## 4. Recover several parameters at once

This is where gradients earn their keep. Searching over dip *and* depth on a
grid would need hundreds of inversions (Chapter 09's checkpoint); the gradient
search converges in a few dozen iterations regardless. We start with both
parameters wrong, dip at 30 degrees and depth at 35 km.

In [ ]:
theta0 = np.array([0.0, 0.0, 35e3, 315.0, 30.0, 180e3, 90e3])   # dip and depth both wrong

result = geodef.invert.geometry_search(
    theta0, gnss,
    ref_lat=-2.0, ref_lon=100.0,
    free=["dip", "depth"],
    bounds={"dip": (5.0, 45.0), "depth": (10e3, 60e3)},
    n_length=n_length, n_width=n_width,
    components="dip",
    regularization="laplacian", regularization_strength=1.0,
)
sigmas = np.sqrt(np.diag(result.theta_cov))
print(f"recovered dip:   {result.theta[4]:7.2f} +/- {sigmas[0]:.2f} degrees  (true 15.0)")
print(f"recovered depth: {result.theta[2] / 1e3:7.2f} +/- {sigmas[1] / 1e3:.2f} km      (true 25.0)")
print(f"reduced chi-squared: {result.reduced_chi2:.3f}")

Both parameters are recovered from a poor joint start. The slip at the
recovered geometry, next to the truth, confirms that geometry and slip together
explain the data.

In [ ]:
best_fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=result.theta[2], strike=315.0, dip=result.theta[4],
    length=180e3, width=90e3, n_length=n_length, n_width=n_width,
)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
vmax = slip_true[N:].max()
geodef.plot.slip(fault, slip_true[N:], ax=ax1, cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="True (dip 15, depth 25 km)",
                 colorbar_label="Slip (m)")
geodef.plot.slip(best_fault, result.slip, ax=ax2, cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax,
                 title=f"Recovered (dip {result.theta[4]:.1f}, "
                       f"depth {result.theta[2] / 1e3:.1f} km)",
                 colorbar_label="Slip (m)")
plt.tight_layout()
plt.show()

## 5. How far to trust the error bars

The search returns a covariance, `theta_cov`, from which we read the error bars.
It is a **Gauss-Newton** covariance: it measures the curvature of the misfit
surface right at the minimum and assumes the surface is a clean, single bowl
there. When that assumption holds, the error bar is a fair local uncertainty.

But Chapter 09 showed that geometry surfaces often have long flat valleys and
sometimes several minima. A local curvature cannot see any of that. It describes
the neighborhood of the one point the optimizer landed on, and nothing beyond.
So the Gauss-Newton error bar is trustworthy only when three things hold: the
minimum is unique (check with a scan or multiple starts, as in Chapter 09), the
misfit is smooth and bowl-shaped nearby, and no bound is active (a parameter
pinned at a bound has a one-sided uncertainty the covariance cannot express).

When those conditions fail, this fast local error bar understates the true
ambiguity. Chapter 14 replaces it with full posterior sampling, which maps the
whole shape of the uncertainty rather than just its curvature at one point.

**Checkpoint.** The very first `geometry_search` call was noticeably slower
than the second. Does that mean the optimization itself got faster on the
second run?

<details><summary>Answer</summary>

No. The first call includes a one-time compilation of the differentiable model
to machine code; the optimization work is similar each time. Later calls with
the same problem shapes skip compilation and reuse the compiled code, so they
start much faster. When you benchmark, separate compile time from run time
rather than reporting them as one number.

</details>

## Checkpoint questions

**What does automatic differentiation remove, and what does it not?**

<details><summary>Answer</summary>

It removes finite-difference truncation error and the extra evaluations needed
to estimate a slope, giving exact gradients cheaply. It does *not* remove the
nonconvexity of the objective: the surface can still have multiple minima and
flat valleys, so a scan or multiple starts is still needed.

</details>

**Why does parameter scaling matter in a multi-parameter search?**

<details><summary>Answer</summary>

The parameters have very different units and ranges: dip is in degrees (tens),
depth in meters (tens of thousands). Without care, the gradient in depth would
dwarf the gradient in dip purely from units, and the optimizer would step
awkwardly. Scaling the parameters to comparable magnitudes keeps the search
well behaved.

</details>

**When is the Gauss-Newton covariance misleading?**

<details><summary>Answer</summary>

When the objective is nonlinear enough that the surface is not a clean bowl,
when there are multiple minima, when a bound is active, or when the noise model
is wrong. In all of these the local curvature does not capture the real spread,
and posterior sampling (Chapter 14) is the honest alternative.

</details>

## Common mistakes

- **Reporting compile and run time as one number.** The first call is dominated
  by compilation; quoting it as "the runtime" hides the fast reuse that makes
  the method worthwhile.
- **Leaving float64 disabled.** Geodetic coordinates involve small differences
  of large numbers. Single precision can lose that signal; keep float64 for
  geometry work.
- **Trusting a fast local optimum without a scan.** Gradients find a minimum
  quickly, but not necessarily the global one. Skipping the basin check of
  Chapter 09 just repeats that mistake faster.

## Recap

- Automatic differentiation gives exact gradients of the misfit for about the
  cost of one forward evaluation, replacing finite differences.
- `geodef.invert.geometry_search` combines variable projection, JAX gradients,
  and L-BFGS-B to recover several geometry parameters at once.
- The first call compiles; later calls reuse the compiled code and are fast.
- Its Gauss-Newton error bars are local: trustworthy only for a unique,
  smooth, interior minimum, and replaced by sampling in Chapter 14.

Chapter 11 leaves planar rectangles behind and puts the same linear inverse
problem onto curved triangular meshes.

## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are in `solutions/10_gradient_geometry_solutions.ipynb`.

1. **More free parameters.** Free three, then five geometry parameters and
   compare the iteration counts and recovered values.
2. **Compile versus run.** Time the first search and a repeated search on the
   same problem, and report the two costs separately.
3. **Scaling.** Change the relative scales of the free parameters and observe
   how the optimizer's behavior changes.
4. **Challenge: check a gradient.** Validate one component of the objective
   gradient against a centered finite-difference estimate, and confirm they
   agree to several digits.

## Further reading

- Griewank, A., and Walther, A. (2008), *Evaluating Derivatives*, on automatic
  differentiation.
- Nocedal, J., and Wright, S. J. (2006), *Numerical Optimization*, on L-BFGS-B
  and Gauss-Newton covariance.
- Golub, G., and Pereyra, V. (1973), on differentiable variable projection.
- [GeoDef gradients documentation](../docs/gradients.md) and
  [inversion documentation](../docs/invert.md) for the differentiable
  interfaces.
- The next course chapter is `11_triangular_faults.ipynb`.